In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND TABLE CONFIGURATION
# =================================================== 

from pyspark.sql import functions as F


"""
Validate persisted Silver operational tables, quarantine traceability,
cross-layer reconciliation, business relationships, and rerun behavior.
"""

DATASETS = {
    "equipment_events": {
        "bronze": "semiconplus_portfolio.bronze.equipment_events",
        "silver": "semiconplus_portfolio.silver.equipment_events",
        "quarantine": "semiconplus_portfolio.quarantine.equipment_events",
        "business_key": "event_id",
    },
    "unit_test_results": {
        "bronze": "semiconplus_portfolio.bronze.unit_test_results",
        "silver": "semiconplus_portfolio.silver.unit_test_results",
        "quarantine": "semiconplus_portfolio.quarantine.unit_test_results",
        "business_key": "test_result_id",
    },
    "tester_logs": {
        "bronze": "semiconplus_portfolio.bronze.tester_logs_raw",
        "silver": "semiconplus_portfolio.silver.tester_logs",
        "quarantine": "semiconplus_portfolio.quarantine.tester_logs",
        "business_key": "lot_id",
    },
}


In [0]:
# ===================================================
# BLOCK 2 — TABLE AVAILABILITY AND RECONCILIATION
# =================================================== 

"""
Confirm that every persisted output exists and that accepted plus rejected
records reconcile exactly to its Bronze source.
"""

validation_counts = []

for dataset_name, config in DATASETS.items():
    for table_name in config.values():
        if "." in table_name:
            assert spark.catalog.tableExists(table_name)

    bronze_count = spark.table(config["bronze"]).count()
    silver_count = spark.table(config["silver"]).count()
    quarantine_count = spark.table(config["quarantine"]).count()

    assert silver_count + quarantine_count == bronze_count

    validation_counts.append(
        (dataset_name, bronze_count, silver_count, quarantine_count)
    )

display(
    spark.createDataFrame(
        validation_counts,
        ["dataset", "bronze", "silver", "quarantine"],
    ).orderBy("dataset")
)

print("Operational cross-layer reconciliation passed.")

In [0]:
# ===================================================
# BLOCK 3 — SILVER KEY AND REQUIRED-FIELD VALIDATION
# =================================================== 

"""
Confirm that accepted datasets contain required identifiers and unique
event or result business keys where the source contract defines them.
"""

for dataset_name, config in DATASETS.items():
    silver_df = spark.table(config["silver"])
    key_column = config["business_key"]

    invalid_key_count = silver_df.filter(
        F.col(key_column).isNull()
        | (F.trim(F.col(key_column)) == "")
    ).count()

    assert invalid_key_count == 0

    # Tester logs contain one record per lot in this initial source snapshot.
    duplicate_key_count = (
        silver_df
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_key_count == 0

print("Operational Silver key validation passed.")

In [0]:
# ===================================================
# BLOCK 4 — EQUIPMENT-EVENT VALIDATION
# =================================================== 

"""
Confirm that accepted equipment events use valid types, states, durations,
and site-equipment relationships.
"""

equipment_df = spark.table(DATASETS["equipment_events"]["silver"])

assert dict(equipment_df.dtypes)["event_timestamp_utc"] == "timestamp"
assert dict(equipment_df.dtypes)["duration_seconds"] == "bigint"

assert equipment_df.filter(F.col("duration_seconds") <= 0).count() == 0

assert equipment_df.filter(
    ~F.col("event_type").isin(
        "RUN", "IDLE", "SETUP", "ALARM", "MAINTENANCE",
        "PLANNED_DOWNTIME", "UNPLANNED_DOWNTIME"
    )
).count() == 0

assert spark.table(DATASETS["equipment_events"]["quarantine"]).filter(
    F.array_contains("_quality_reasons", "INVALID_DURATION_SECONDS")
).count() == 1

print("Equipment-event validation passed.")

In [0]:
# ===================================================
# BLOCK 5 — UNIT-TEST VALIDATION
# =================================================== 

"""
Confirm that accepted unit-test results are fully parsed and comply with
the declared grain, test-status domain, and measurement controls.
"""

tests_df = spark.table(DATASETS["unit_test_results"]["silver"])

assert dict(tests_df.dtypes)["event_timestamp_utc"] == "timestamp"
assert dict(tests_df.dtypes)["unit_sequence"] == "int"
assert dict(tests_df.dtypes)["test_time_seconds"] == "double"

assert tests_df.filter(
    ~F.col("test_status").isin("PASS", "FAIL")
).count() == 0

assert tests_df.filter(F.col("test_time_seconds") <= 0).count() == 0

assert tests_df.filter(
    F.col("lot_id").isNull()
    | F.col("device_id").isNull()
    | F.col("equipment_id").isNull()
).count() == 0

print("Unit-test-result validation passed.")

In [0]:
# ===================================================
# BLOCK 6 — TESTER-LOG VALIDATION
# =================================================== 

"""
Confirm that accepted tester logs were parsed successfully and that their
production quantities reconcile.
"""

logs_df = spark.table(DATASETS["tester_logs"]["silver"])

assert dict(logs_df.dtypes)["event_timestamp_utc"] == "timestamp"
assert dict(logs_df.dtypes)["quantity_started"] == "bigint"

assert logs_df.filter(
    F.col("event_timestamp_utc").isNull()
    | F.col("lot_id").isNull()
    | F.col("device_id").isNull()
    | F.col("equipment_id").isNull()
).count() == 0

assert logs_df.filter(
    F.col("quantity_passed") + F.col("quantity_failed")
    != F.col("quantity_started")
).count() == 0

print("Tester-log validation passed.")

In [0]:
# ===================================================
# BLOCK 7 — QUARANTINE TRACEABILITY
# =================================================== 

"""
Confirm that every rejected record retains at least one quality reason,
source lineage, and pipeline-run metadata.
"""

for dataset_name, config in DATASETS.items():
    quarantine_df = spark.table(config["quarantine"])

    invalid_quarantine_count = quarantine_df.filter(
        F.col("_quality_reasons").isNull()
        | (F.size("_quality_reasons") == 0)
        | F.col("_source_file_path").isNull()
        | F.col("_bronze_pipeline_run_id").isNull()
        | F.col("_silver_pipeline_run_id").isNull()
    ).count()

    assert invalid_quarantine_count == 0

print("Operational quarantine traceability passed.")

In [0]:
# ===================================================
# BLOCK 8 — CAPTURE RERUN BASELINE
# =================================================== 

"""
Capture current Silver and quarantine counts before repeating the
deterministic operational-data transformation.
"""

counts_before_rerun = {
    dataset_name: (
        spark.table(config["silver"]).count(),
        spark.table(config["quarantine"]).count(),
    )
    for dataset_name, config in DATASETS.items()
}

print(counts_before_rerun)

In [0]:
# ---------------
# RERUN PROCEDURE
# ---------------

# 1. Run Block 8.
# 2. Rerun all blocks in 09_silver_operational_data.
# 3. Return without clearing this notebook session.
# 4. Run Block 9.

# ===================================================
# BLOCK 9 — IDEMPOTENCY VALIDATION
# =================================================== 

"""
Confirm that repeating the transformation replaces current snapshots
without changing accepted or quarantined record counts.
"""

counts_after_rerun = {
    dataset_name: (
        spark.table(config["silver"]).count(),
        spark.table(config["quarantine"]).count(),
    )
    for dataset_name, config in DATASETS.items()
}

assert counts_after_rerun == counts_before_rerun

print("SILVER OPERATIONAL-DATA IDEMPOTENCY TEST PASSED")

In [0]:

# ===================================================
# BLOCK 10 — FINAL VALIDATION RESULT
# =================================================== 

"""
Publish the final validation result for project documentation and
execution evidence.
"""

print("SILVER OPERATIONAL-DATA VALIDATION PASSED")
for row in validation_counts:
    print(
        f"{row[0]}: bronze={row[1]:,}, "
        f"silver={row[2]:,}, quarantine={row[3]:,}"
    )